# Step 8 - Annotation Schema and Annotation Sample

Purpose: define the human annotation scheme for the two project models (implicit/explicit classifier and rewrite recommender) and draw the sample that annotators will label.

This notebook does **not** train anything. Step 6 rule flags are used only as *sampling strata*, never as ground truth. Ground truth comes from the human columns exported here.

Design basis: a three-layer scheme (bias presence, bias type, protected attribute) with a span layer, following the structure of JABD (explicit/implicit, span tags, job ads), Frissen et al. (2023), Nbias (Raza et al., 2024), the hierarchical labels of SemEval-2023 Task 10 (Kirk et al.), and multi-label spans in GUS-Net.

Inter-annotator agreement is **out of scope for this phase**; it will be reconsidered in the next phase. The sample therefore has a single annotator column set.

Inputs: `step4_sentence_units.csv` (7,971 canonical units with `employer_id`) and `step6_sentence_level.csv`. Outputs (`outputs/equal_step8_annotation/`): `step8_annotation_schema.json`, `step8_annotation_sample.csv`, `step8_sampling_summary.csv`.

## 1. Annotation schema

Unit of annotation: one sentence/requirement unit (Step 2). The annotator also sees the full ad for context.

| Field | Values | Rule |
|---|---|---|
| `bias_type` | `explicit`, `implicit`, `neutral_statement`, `none`, `unclear` | `explicit`: the unit **states** a restriction or preference on a protected attribute (male only, female only, "Female/Male", gendered title or pronoun, age limit, civil-status requirement). `implicit`: no stated restriction, but coded wording or a proxy that can discourage a group (agentic/communal coded traits, appearance or physical requirements). `neutral_statement`: an explicit inclusion such as "Open to all qualified candidates". `none`: neither. `unclear`: cannot be decided from the unit and its ad. |
| `bias_presence` | derived, not annotated | `biased` if `bias_type` is explicit or implicit; `neutral` if `neutral_statement` or `none`; `unclear` rows are excluded. |
| `attribute` | multi-label, `;`-separated: `gender`, `age`, `marital_status`, `appearance_physical`, `education_socioeconomic`, `ethnicity`, `religion`, `disability`, `other`, `none` | Required for explicit and implicit; `none` otherwise. Several values are allowed (e.g. "pleasing personality, female and single" = `gender;marital_status;appearance_physical`). Use `education_socioeconomic` only for status wording such as "graduate of a reputable university"; an ordinary degree requirement ("Graduate of BS Psychology") is a job requirement, not bias. |
| `explicit_direction` | `male`, `female`, `both_listed`, `title_or_pronoun`, `na` | Only when `bias_type = explicit` and `attribute` contains `gender`. "Female/Male" or "Male/Female" is `both_listed` and still explicit, per the project decision. |
| `gender_coding` | `agentic`, `communal`, `na` | Only when `bias_type = implicit` and `attribute` contains `gender`. |
| `span_text` | exact substring(s) of the unit, separated by ` \|\| ` | The longest continuous phrase that carries the bias. Required for explicit and implicit. If the whole unit is biased, copy the whole unit. |
| `language_quality_flag` | `yes`, `no` | `yes` when the unit has a typo, malapropism or non-standard grammar (e.g. "faucet" for "facets"). This flag is recorded **separately** and must never change `bias_type`: grammar errors are not bias (Harris et al., 2022). |
| `context_dependent` | `yes`, `no` | `yes` when the label would change without the surrounding ad (e.g. "male" describing clients, not applicants). |
| `rewrite_target` | free text | Neutral rewrite of the unit, required for explicit and implicit. Change only the biased wording and keep education, experience and other requirements. An explicit gender, age or civil-status requirement is removed or replaced by "Open to all qualified candidates". Coded words are replaced by neutral wording, not by opposite-gender-coded wording. |
| `notes` | free text | Reasoning for hard cases. |

In [11]:
import json
from pathlib import Path

import pandas as pd

SEED = 42
OUT = Path("outputs") / "equal_step8_annotation"
OUT.mkdir(parents=True, exist_ok=True)

SCHEMA = {
    "bias_type": ["explicit", "implicit", "neutral_statement", "none", "unclear"],
    "attribute": ["gender", "age", "marital_status", "appearance_physical", "education_socioeconomic", "ethnicity", "religion", "disability", "other", "none"],
    "explicit_direction": ["male", "female", "both_listed", "title_or_pronoun", "na"],
    "gender_coding": ["agentic", "communal", "na"],
    "language_quality_flag": ["yes", "no"],
    "context_dependent": ["yes", "no"],
}
(OUT / "step8_annotation_schema.json").write_text(json.dumps(SCHEMA, indent=2), encoding="utf-8")

units = pd.read_csv("outputs/equal_step4_sentence_embeddings/step4_sentence_units.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
flags = pd.read_csv("outputs/equal_step6_bias_analysis/step6_sentence_level.csv", encoding="utf-8-sig", dtype={"sentence_id": str})
flags["explicit_unit_type"] = flags["explicit_unit_type"].fillna("")
df = units.merge(flags[["sentence_id", "explicit_unit_type", "masc_hits", "fem_hits"]], on="sentence_id", how="left", validate="one_to_one")
assert len(df) == 7971 and df["masc_hits"].notna().all()
print("units:", len(df), "| ads:", df["record_id"].nunique(), "| employers:", df["employer_id"].nunique())

units: 7971 | ads: 1500 | employers: 320


## 2. Sampling strata

Candidate pools come from Step 6 flags plus three keyword patterns (marital/civil status, age, appearance). These only help find candidates for the annotator; they are not labels. Age and civil-status limits are now sampled as **explicit** candidates, and appearance as an **implicit** candidate. A `control` stratum of unflagged units lets the annotator find bias the rules miss. Priority when a unit matches several: restrictive > inclusive > marital > age > appearance > coded > control.

In [12]:
PROXY = {
    "proxy_marital": r"\b(?:single|unmarried|married|marital|civil status)\b",
    "proxy_age": r"(?:\b\d{2}\s*(?:to|-|\u2013)\s*\d{2}\b|\byears?\s+old\b|\bage[d]?\b|\b(?:below|under|not more than|maximum of)\s+\d{2}\b)",
    "proxy_appearance": r"\b(?:pleasing personality|good[- ]looking|height|attractive|presentable|physically fit)\b",
}
low = df["sentence"].str.lower()
df["stratum"] = "control"
df.loc[df["fem_hits"].astype(int) > 0, "stratum"] = "coded_feminine"
df.loc[df["masc_hits"].astype(int) > 0, "stratum"] = "coded_masculine"
for name in ["proxy_appearance", "proxy_age", "proxy_marital"]:
    df.loc[low.str.contains(PROXY[name], regex=True), "stratum"] = name
df.loc[df["explicit_unit_type"] == "inclusive", "stratum"] = "inclusive_wording"
df.loc[df["explicit_unit_type"] == "restrictive_female", "stratum"] = "restrictive_female"
df.loc[df["explicit_unit_type"] == "restrictive_male", "stratum"] = "restrictive_male"
pool = df["stratum"].value_counts().rename("pool_size")
display(pool.to_frame())

,pool_size
stratum,
control,6775
coded_masculine,308
coded_feminine,269
restrictive_male,208
restrictive_female,113
inclusive_wording,102
proxy_appearance,90
proxy_age,65
proxy_marital,41


## 3. Draw the sample

Target sizes per stratum, capped at 3 units per employer per stratum so that no single employer dominates (Step 6 showed one employer holds 30% of title/pronoun ads). Strata smaller than the target are taken in full. Sampling is seeded.

In [13]:
TARGET = {"restrictive_male": 60, "restrictive_female": 60, "inclusive_wording": 102, "proxy_marital": 40, "proxy_age": 40,
          "proxy_appearance": 60, "coded_masculine": 80, "coded_feminine": 80, "control": 100}
PER_EMPLOYER_CAP = 3

parts = []
for stratum, n in TARGET.items():
    g = df[df["stratum"] == stratum].sample(frac=1, random_state=SEED)
    g = g.groupby("employer_id", sort=False).head(PER_EMPLOYER_CAP)
    parts.append(g.head(n))
sample = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

ad_text = df.sort_values("sentence_id").groupby("record_id")["sentence"].apply(lambda s: " | ".join(s))
sample["ad_context"] = sample["record_id"].map(ad_text)

LABEL_COLS = ["bias_type", "attribute", "explicit_direction", "gender_coding", "span_text", "language_quality_flag", "context_dependent", "rewrite_target", "notes"]
for c in LABEL_COLS:
    sample[c] = ""

keep = ["sentence_id", "record_id", "employer_id", "component", "sentence", "ad_context", "stratum"] + LABEL_COLS
sample[keep].to_csv(OUT / "step8_annotation_sample.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame({"pool_size": pool, "target": pd.Series(TARGET), "sampled": sample["stratum"].value_counts()}).fillna(0).astype(int)
summary["employers_in_sample"] = sample.groupby("stratum")["employer_id"].nunique()
summary.to_csv(OUT / "step8_sampling_summary.csv", encoding="utf-8-sig")
display(summary)
print("sample units:", len(sample), "| ads:", sample["record_id"].nunique(), "| employers:", sample["employer_id"].nunique(),
      "| duplicate sentence_id:", int(sample["sentence_id"].duplicated().sum()), "| distinct unit texts:", sample["sentence"].nunique())

,pool_size,target,sampled,employers_in_sample
coded_feminine,269,80,80,52
coded_masculine,308,80,80,54
control,6775,100,100,66
inclusive_wording,102,102,86,52
proxy_age,65,40,40,25
proxy_appearance,90,60,60,41
proxy_marital,41,40,30,18
restrictive_female,113,60,60,49
restrictive_male,208,60,60,39


sample units: 596 | ads: 469 | employers: 208 | duplicate sentence_id: 0 | distinct unit texts: 310


## 4. Worked examples from our data

Rule-based stratum shown beside each unit. The annotator decides the actual label; the stratum may be wrong.

In [14]:
pd.set_option("display.max_colwidth", 140)
ex = sample.groupby("stratum", sort=False).head(2).sort_values("stratum")
display(ex[["stratum", "sentence_id", "sentence"]].reset_index(drop=True))

,stratum,sentence_id,sentence
0,coded_feminine,row-0751-s008,Good interpersonal and presentation skills
1,coded_feminine,row-0914-s005,"Honest, hand working and organized"
2,coded_masculine,row-0860-s005,Must be keen to details and good analytical skills
3,coded_masculine,row-0383-s002,Vocational graduate and/or technical course background with at least 2 years of experience leading the entire production team or with si...
4,control,row-0766-s003,Must be a college graduate of BS Accountancy or Accounting tech
5,control,row-0095-s009,With or without experience
6,inclusive_wording,row-0898-s002,Male or Female
7,inclusive_wording,row-1434-s002,Male or Female
8,proxy_age,row-0012-s003,20 Years old Above
9,proxy_age,row-1304-s003,Must be 18 years old and above


## 5. Label validator (run after annotation)

Checks that every filled row uses allowed values, that `span_text` is an exact substring of the unit, that explicit/implicit rows have an attribute, span and rewrite, and that `language_quality_flag` is filled. It then derives `bias_presence`. While the label columns are empty it only reports that no labels exist.

In [15]:
def validate(path=OUT / "step8_annotation_sample.csv"):
    d = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    d = d[d["bias_type"] != ""].copy()
    if d.empty:
        return d, ["no human labels yet"]
    errors = []
    for i, r in d.iterrows():
        sid = r["sentence_id"]
        if r["bias_type"] not in SCHEMA["bias_type"]:
            errors.append((sid, "bias_type"))
        attrs = [a for a in r["attribute"].split(";") if a]
        if any(a not in SCHEMA["attribute"] for a in attrs):
            errors.append((sid, "attribute"))
        biased = r["bias_type"] in ("explicit", "implicit")
        if biased and (not attrs or attrs == ["none"] or not r["span_text"] or not r["rewrite_target"]):
            errors.append((sid, "missing attribute/span/rewrite"))
        if any(s and s not in r["sentence"] for s in r["span_text"].split(" || ")):
            errors.append((sid, "span not in unit"))
        if r["language_quality_flag"] not in SCHEMA["language_quality_flag"]:
            errors.append((sid, "language_quality_flag"))
    d["bias_presence"] = d["bias_type"].map({"explicit": "biased", "implicit": "biased", "neutral_statement": "neutral", "none": "neutral", "unclear": "unclear"})
    return d, errors

labelled, errs = validate()
print("labelled units:", len(labelled), "| validation errors:", errs)

labelled units: 0 | validation errors: ['no human labels yet']
